# 67. 구조화 조회의 대상 표현과 추출기 비교

67A는 저장된 144개 추출 계획을 고정하고 조회 규칙만 비교한다. 67B는 선택한 DB·매칭 규칙을 고정한 뒤 기존·수정 추출기를 새로 비교한다. 개발 평가이며 운영 코드·원문·기존 DB는 변경하지 않는다.

공통 SQL 연결은 `scripts/67_query_repair.py`로 재사용한다. 노트북이 실험 조건·입출력·선택을 소유하며, helper는 68·69에서도 같은 조회 규칙을 쓰기 위해 둔다. HTML은 결과 검토 후 직접 작성한다.

In [1]:
from pathlib import Path
import copy, importlib.util, json, sqlite3, time
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
spec = importlib.util.spec_from_file_location('query67_notebook', ROOT/'notebooks/scripts/67_query_repair.py')
K = importlib.util.module_from_spec(spec); spec.loader.exec_module(K)
R = K.load_runtime()
from hashlib import sha256
BASE = ROOT/'notebooks/data/67_structured_retrieval_repair'
BASE.mkdir(exist_ok=True)
read = lambda p: json.loads(Path(p).read_text())
def save(name, value):
    path = BASE/name
    if path.exists():
        assert read(path) == value, '기존 결과와 다름: '+name
    else:
        path.write_text(json.dumps(value, ensure_ascii=False, indent=2)+'\n')
    return value
sha = lambda p: sha256(Path(p).read_bytes()).hexdigest()
directory = R.M.DB_NEW
facts = R.R.fact_map(directory)
documents = [json.loads(s) for s in (directory/'documents.jsonl').read_text().splitlines()]
plans = read(R.B65/'routing_diagnostics.json')['records']
previous = read(R.B65/'retrieval_results.json')['records']
old_by_query_repeat = {(x['query_id'], x['repeat']): x for x in previous if x['mode']=='SEQ'}
assert len(plans) == 144
print('저장 계획', len(plans), '카드', len(documents), '혜택행', len(facts), '신규 API 0')

저장 계획 144 카드 106 혜택행 3403 신규 API 0


## 67A: 원자료와 DB 보존·일치 확인

현재 65번 기준 DB는 60번에서 정규화한 `db_v3`다. 여기서는 DB 값을 바꾸지 않고 읽기 전용으로 연다. SQLite에 저장된 원본 JSON과 정규화 필드가 JSONL 내보내기와 같은지 모든 행을 확인한다. 이는 PDF 전체 추출 정확도의 증명과는 다르다.

In [2]:
bound_paths = [directory/name for name in ('structured.sqlite', 'facts.jsonl', 'documents.jsonl')]
bound_paths += [R.B65/name for name in ('routing_diagnostics.json', 'parser_results.json', 'retrieval_results.json')]
bound_paths += [ROOT/'notebooks/scripts'/name for name in ('67_query_repair.py', '67_test_query_repair.py', '60_query_repair.py', '65_scope_repair.py')]
inputs = {str(p.relative_to(ROOT)): sha(p) for p in bound_paths}
save('67A_input_contract.json', {'stage':'saved_plan_replay', 'files_sha256':inputs,
    'matcher_versions':['baseline60_literal', K.VERSION], 'DB_changed':False,
    'notebook_phase_code_sha256':sha256(json.dumps([(c['id'],c['source']) for c in read(ROOT/'notebooks/67_structured_retrieval_repair.ipynb')['cells'] if c['cell_type']=='code' and c['id'].startswith('exp67-A-')],ensure_ascii=False,sort_keys=True).encode()).hexdigest(),
    'new_API_calls':0, 'queries':48, 'repeats':3, 'plans':144,
    'candidate_rule_chosen_after_observing_development_zero_cases':True,
    'caution':'후행 개발 규칙이며 별도 69 확인 전 일반화·운영 채택 근거가 아니다.'})
with sqlite3.connect((directory/'structured.sqlite').resolve().as_uri()+'?mode=ro&immutable=1', uri=True) as database:
    database.execute('PRAGMA query_only=ON')
    rows = database.execute('SELECT fact_id,document_id,raw_json,derived_json FROM facts ORDER BY fact_id').fetchall()
    assert len(rows) == len(facts)
    assert {r[0] for r in rows} == set(facts)
    for fid, card, raw, derived in rows:
        assert facts[fid]['document_id'] == card
        assert json.loads(raw) == facts[fid]['raw_fact']
        assert json.loads(derived) == facts[fid]['derived']
save('67A_DB_export_consistency.json', {'status':'all_facts_equal', 'facts':len(rows),
    'raw_fact_equal':True, 'derived_equal':True, 'card_owner_equal':True,
    'source_files_sha256':{str(p.relative_to(ROOT)):sha(p) for p in bound_paths[:3]},
    'limitation':'추출 JSON의 표·공통조건 연결 정확도 자체는 이 등가 검사로 증명하지 않는다.'})
print('SQLite와 JSONL 동일', len(rows), '행')

SQLite와 JSONL 동일 3403 행


## 고정 계획의 전체 재생

추출 결과를 다시 생성하지 않는다. 같은 저장 계획·DB·발급사/카드 범위에서 기존 규칙과 좁은 대상 후보 보완만 바꾼다. ‘진료비’, ‘회비’ 뒤가 아닌 앞의 명시 대상도 후보로 허용한다. 제외·숫자·같은 카드 AND는 유지한다. 기차→철도, 극장→영화 같은 새 동의어 보정은 하지 않는다.

카드 범위를 줄인 뒤의 비정형 검색은 아직 이 단계의 측정 대상이 아니다. 아래 시간은 로컬 구조화 SQL 조회 시간이다.

In [3]:
replay_path = BASE/'67A_replay_results.json'
if replay_path.exists():
    replay = read(replay_path)
    assert replay['input_contract_sha256'] == sha(BASE/'67A_input_contract.json')
else:
    comparisons=[]; started=time.perf_counter(); cpu=time.process_time()
    for index, record in enumerate(plans):
        qid, repeat = record['query_id'], record['repeat']
        original = R.S.original_scope(record['question'], documents)
        assert original == old_by_query_repeat[qid, repeat]['original_scope']
        outputs={}
        order = ['baseline','projected'] if index%2 else ['projected','baseline']
        for matcher in order:
            outputs[matcher] = K.route(R, record['plan'], original, directory, facts,
                matcher=matcher, parser_failed=record['parser_status']!='ok')
        old_cards = set(old_by_query_repeat[qid, repeat]['routing']['eligible_cards'])
        assert set(outputs['baseline']['eligible_cards']) == old_cards
        new_cards=set(outputs['projected']['eligible_cards'])
        comparisons.append({'query_id':qid,'repeat':repeat,'question':record['question'],
            'original_scope':original,'saved_plan':record['plan'],'outputs':outputs,
            'added_cards':sorted(new_cards-old_cards),'lost_cards':sorted(old_cards-new_cards),
            'previous_positive':bool(old_cards),'changed_cards':old_cards!=new_cards})
        if (index+1)%24==0: print('재생',index+1,'/144')
    replay={'status':'replayed_awaiting_manual_candidate_audit', 'records':comparisons,
        'input_contract_sha256':sha(BASE/'67A_input_contract.json'),
        'wall_seconds':time.perf_counter()-started,'cpu_seconds':time.process_time()-cpu,
        'new_API_calls':0}
    save('67A_replay_results.json', replay)
for name, expected in inputs.items(): assert sha(ROOT/name) == expected
records=replay['records']
print('기존 양성',sum(r['previous_positive'] for r in records),
      '보완 후 양성',sum(bool(r['outputs']['projected']['eligible_cards']) for r in records),
      '변경',sum(r['changed_cards'] for r in records),
      '기존 양성 카드손실',sum(bool(r['lost_cards']) for r in records))

기존 양성 100 보완 후 양성 108 변경 8 기존 양성 카드손실 0


## 수동 정답 계획 진단 및 추출기 후속 비교

기존 0건의 원인을 대상 입도·행위·정규화·관계·실제 부재로 나누고, 양성 계획도 변경된 카드와 조건을 검토한다. 수동 계획은 DB 표현 가능성 진단용이며 자동 검색·답변의 입력으로 쓰지 않는다.

67B의 신규 추출 비교 셀은 66 평가 완료와 67A 후보 검토 후 추가한다. 현재 결과는 조회 규칙의 준비·개발 재생 단계이며 최종 설정 선택이 아니다.

## 67A 보완: 같은 행의 정규 실적 수치 정규화

문자 대상 문제와 별개로, 조건 본문에는 정규 전월실적이 있지만 수치 필드가 `unknown`인 행을 조사한다. 여러 금액 중 거래금액과 전월실적 금액을 구분하지 못하거나 ‘전월 20만원 이상’ 같은 표현이 누락된 경우다.

실험 DB 사본에서만, 같은 행의 명시 문구·타입 금액·조건 출처가 일치하는 단일 `이상` 조건을 보완한다. 범위·신규 유예·연간/복수 기간·엄격 초과·미기재는 보완하지 않는다. 원본 JSON 필드·관계·기존 known/0원은 보존하며 공통조건을 다른 행으로 추측 전파하지 않는다. OCR 및 JSON의 행 연결을 신뢰한다는 기존 실험 전제 아래의 수치 파생값 수정이다.

In [4]:
import resource, shutil
spec = importlib.util.spec_from_file_location('numeric67_notebook', ROOT/'notebooks/scripts/67_db_repair.py')
D = importlib.util.module_from_spec(spec); spec.loader.exec_module(D)
prototype = BASE/'db_v4'
patches = [patch for fact in facts.values() if (patch := D.spend_patch(fact))]
patch_by_id = {p['fact_id']:p for p in patches}
db_phase_sources = {**inputs, **{str((ROOT/'notebooks/scripts'/name).relative_to(ROOT)):sha(ROOT/'notebooks/scripts'/name)
    for name in ('67_db_repair.py','67_test_db_repair.py')}}
db_code = [(c['id'],c['source']) for c in read(ROOT/'notebooks/67_structured_retrieval_repair.ipynb')['cells']
           if c['cell_type']=='code' and c['id'].startswith('exp67-DB-')]
db_contract={'input_files_sha256':db_phase_sources,'normalization_rule':D.VERSION,
    'notebook_phase_code_sha256':sha256(json.dumps(db_code,ensure_ascii=False,sort_keys=True).encode()).hexdigest(),
    'patches':patches,'OCR_and_JSON_row_relations_assumed_valid':True,
    'not_source_table_relationship_validation':True,'new_API_calls':0}
save('67A_DB_repair_contract.json',db_contract)
manifest_path=BASE/'67A_DB_copy_manifest.json'
if manifest_path.exists():
    copy_manifest=read(manifest_path)
    assert copy_manifest['repair_contract_sha256']==sha(BASE/'67A_DB_repair_contract.json')
    for name,expected in copy_manifest['output_files_sha256'].items():assert sha(ROOT/name)==expected
else:
    assert not prototype.exists(),'불완전한 신규 사본을 먼저 확인하세요. 기존 원본은 수정하지 않습니다.'
    started=time.perf_counter(); cpu=time.process_time();prototype.mkdir()
    modified=[]
    for fact in facts.values():
        new=copy.deepcopy(fact)
        if fact['fact_id'] in patch_by_id:new['derived'].update(patch_by_id[fact['fact_id']]['updates'])
        modified.append(new)
    (prototype/'facts.jsonl').write_text(''.join(json.dumps(f,ensure_ascii=False,sort_keys=True)+'\n' for f in modified))
    shutil.copyfile(directory/'documents.jsonl',prototype/'documents.jsonl')
    with sqlite3.connect((directory/'structured.sqlite').resolve().as_uri()+'?mode=ro&immutable=1',uri=True) as old_db, sqlite3.connect(prototype/'structured.sqlite') as new_db:
        old_db.backup(new_db)
        before={row[0]:row for row in old_db.execute('SELECT * FROM facts')}
        columns=[x[1]for x in old_db.execute('PRAGMA table_info(facts)')]
        by_id={f['fact_id']:f for f in modified}
        for patch in patches:
            d=by_id[patch['fact_id']]['derived']
            new_db.execute('UPDATE facts SET spend_threshold_krw=?,spend_operator=?,spend_state=?,derived_json=? WHERE fact_id=?',
                (d['spend_threshold_krw'],d['spend_operator'],d['spend_state'],json.dumps(d,ensure_ascii=False,sort_keys=True),patch['fact_id']))
        new_db.execute('INSERT INTO metadata(key,value_json) VALUES (?,?)',('normalization67',json.dumps({'version':D.VERSION,'source_DB_sha256':sha(directory/'structured.sqlite'),'patched_facts':len(patches)},sort_keys=True)))
        new_db.commit()
        allowed={'spend_threshold_krw','spend_operator','spend_state','derived_json'}
        for row in new_db.execute('SELECT * FROM facts'):
            changed={columns[i] for i,(a,b)in enumerate(zip(before[row[0]],row))if a!=b}
            assert changed<=allowed
            assert bool(changed)==(row[0]in patch_by_id)
            f=by_id[row[0]]
            assert json.loads(row[columns.index('raw_json')])==f['raw_fact']
            assert json.loads(row[columns.index('derived_json')])==f['derived']
        for table in ('documents','target_aliases'):
            assert old_db.execute('SELECT * FROM '+table+' ORDER BY 1').fetchall()==new_db.execute('SELECT * FROM '+table+' ORDER BY 1').fetchall()
        assert old_db.execute("SELECT type,name,tbl_name,sql FROM sqlite_master ORDER BY name").fetchall()==new_db.execute("SELECT type,name,tbl_name,sql FROM sqlite_master ORDER BY name").fetchall()
        assert new_db.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
        assert new_db.execute('PRAGMA foreign_key_check').fetchall()==[]
    old_zero={f['fact_id']for f in facts.values()if f['derived']['spend_state']=='known'and f['derived']['spend_threshold_krw']==0}
    new_zero={f['fact_id']for f in modified if f['derived']['spend_state']=='known'and f['derived']['spend_threshold_krw']==0}
    assert old_zero==new_zero
    for original,new in zip(facts.values(),modified):
        assert {k:v for k,v in original.items()if k!='derived'}=={k:v for k,v in new.items()if k!='derived'}
        allowed_derived={'spend_threshold_krw','spend_operator','spend_state','spend_reason'}
        assert {k:v for k,v in original['derived'].items()if k not in allowed_derived}=={k:v for k,v in new['derived'].items()if k not in allowed_derived}
    for name,expected in db_phase_sources.items():assert sha(ROOT/name)==expected
    copy_manifest={'status':'prototype_copy_built_original_preserved','repair_contract_sha256':sha(BASE/'67A_DB_repair_contract.json'),
        'facts':len(modified),'changed_facts':len(patches),'changed_cards':len({p['document_id']for p in patches}),
        'raw_fact_and_relationship_fields_unchanged':True,'known_zero_fact_ids_preserved':True,'schema_unchanged':True,
        'documents_and_aliases_unchanged':True,'SQLite_JSONL_derived_consistent':True,
        'output_files_sha256':{str((prototype/name).relative_to(ROOT)):sha(prototype/name)for name in ('structured.sqlite','facts.jsonl','documents.jsonl')},
        'wall_seconds':time.perf_counter()-started,'cpu_seconds':time.process_time()-cpu,
        'process_peak_RSS_KiB':resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,'new_API_calls':0}
    save('67A_DB_copy_manifest.json',copy_manifest)
print('새 사본의 실적 파생값 보완',copy_manifest['changed_facts'],'행 /',copy_manifest['changed_cards'],'카드')

새 사본의 실적 파생값 보완 166 행 / 35 카드


## DB × 대상 매칭의 교차 재생

같은 144개 저장 계획으로 기존 DB/새 DB와 기존 매칭/대상 후보 보완을 모두 조합해 4조건을 비교한다. 새로운 LLM 추출은 아직 하지 않는다. 수치 파생값과 대상 매칭의 효과를 분리하고 기존 양성의 추가/손실 카드까지 확인한다.

In [5]:
factorial_path=BASE/'67A_DB_matcher_factorial.json'
if factorial_path.exists():
    factorial=read(factorial_path)
    assert factorial['repair_contract_sha256']==sha(BASE/'67A_DB_repair_contract.json')
else:
    f4=R.R.fact_map(prototype);all_comparisons=[];started=time.perf_counter();cpu=time.process_time()
    arms=[('DB3_baseline',directory,facts,'baseline'),('DB3_projected',directory,facts,'projected'),
          ('DB4_baseline',prototype,f4,'baseline'),('DB4_projected',prototype,f4,'projected')]
    for index,record in enumerate(plans):
        original=R.S.original_scope(record['question'],documents);outputs={};shift=index%4
        for label,db,fm,matcher in arms[shift:]+arms[:shift]:
            outputs[label]=K.route(R,record['plan'],original,db,fm,matcher=matcher,parser_failed=record['parser_status']!='ok')
        original_cards=set(old_by_query_repeat[record['query_id'],record['repeat']]['routing']['eligible_cards'])
        assert set(outputs['DB3_baseline']['eligible_cards'])==original_cards
        all_comparisons.append({'query_id':record['query_id'],'repeat':record['repeat'],'question':record['question'],
            'saved_plan':record['plan'],'original_scope':original,'outputs':outputs,
            'card_changes':{label:{'added':sorted(set(result['eligible_cards'])-original_cards),
                                  'lost':sorted(original_cards-set(result['eligible_cards']))}for label,result in outputs.items()}})
        if (index+1)%24==0:print('4条件 재생',index+1,'/144')
    factorial={'status':'replayed_awaiting_query_level_candidate_audit','records':all_comparisons,
        'repair_contract_sha256':sha(BASE/'67A_DB_repair_contract.json'),
        'wall_seconds':time.perf_counter()-started,'cpu_seconds':time.process_time()-cpu,'new_API_calls':0}
    save('67A_DB_matcher_factorial.json',factorial)
for name,expected in db_phase_sources.items():assert sha(ROOT/name)==expected
for name,expected in copy_manifest['output_files_sha256'].items():assert sha(ROOT/name)==expected
for label in ('DB3_baseline','DB3_projected','DB4_baseline','DB4_projected'):
    print(label,'양성계획',sum(bool(r['outputs'][label]['eligible_cards'])for r in factorial['records']),
          '카드목록변경',sum(bool(r['card_changes'][label]['added']or r['card_changes'][label]['lost'])for r in factorial['records']),
          '기존카드손실',sum(bool(r['card_changes'][label]['lost'])for r in factorial['records']))

4条件 재생 24 /144
4条件 재생 48 /144
4条件 재생 72 /144
4条件 재생 96 /144
4条件 재생 120 /144
4条件 재생 144 /144
DB3_baseline 양성계획 100 카드목록변경 0 기존카드손실 0
DB3_projected 양성계획 108 카드목록변경 8 기존카드손실 0
DB4_baseline 양성계획 100 카드목록변경 0 기존카드손실 0
DB4_projected 양성계획 108 카드목록변경 8 기존카드손실 0
